In [ ]:
import pathlib
# from typing import List

import yaml
# import numpy as np
import pandas as pd
from IPython.display import display

# import jax
# import jax.numpy as jnp
import jax.random as jr
# import optax
# import equinox as eqx
# import numpyro.distributions as dist

# import torch

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

# import ipdb

# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

In [ ]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [ ]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

In [ ]:
parameters = {
    'training': {
        'train_test_split': {
            'test_size':  0.2,
            # 'random_state': 42
        },
        'batch_size': 32,
        # 'learning_rate': 1e-3,
        'learning_rate': 3e-5,
        # 'learning_rate': 1e-6,
        'dl_rng_key': 43,
    },
    'PRNG seeds': {
        'training-validation splitting': 42,
        'training dataloader': 43,
        'model initialization': {
            'encoder': 42,
            'decoder': 43,
        },
        'sampling': {
            'training set': 14,
            'validation set': 15,
        },
    },
}

In [ ]:
# train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(ds, parameters['training']['splitting'], parameters['training']['batch_size'], parameters['training']['dl_rng_key'])
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])

In [ ]:
batch = next(iter(train_dl))

In [ ]:
signals_len = ds[0][1].shape[-1]
signals_len

In [ ]:
tuple(ds[0][1].shape)

In [ ]:
parameters['encoder'] = {
    'input_size': list(ds[0][1].shape),
}

In [ ]:
parameters['model'] = {
    # 'dim_z': 4,
    'dim_z': 10,
    # 'dim_z': 20,
    'kernel_sizes': [128, 64, 32],
    'channels': [12, 12, 12],
    'stride': 2,
}

In [ ]:
x = batch[1][0].numpy()
encoder = trapallada.model.signal.Encoder(
    **(parameters['model'] | parameters['encoder']),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

In [ ]:
encoder.layers_spatial_sizes

In [ ]:
z = encoder(x)
z.shape

In [ ]:
# parameters['decoder'] = {'layers_spatial_sizes': encoder.layers_spatial_sizes}

In [ ]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
# decoder = trapallada.model.signal.Decoder(**(parameters['model'] | parameters['decoder']), key=jr.PRNGKey(42))
decoder = trapallada.model.signal.Decoder(
    **(parameters['model'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

In [ ]:
vae = trapallada.model.signal.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

In [ ]:
x_pred

In [ ]:
parameters['training']['n_epochs'] = 3
parameters['loss'] = {'x_sd': 0.5}

In [ ]:
output_file = pathlib.Path('parameters').with_suffix('.yaml')
with open(output_file, 'w') as f:
    yaml.dump(parameters, f, default_flow_style=False)

In [ ]:
model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)

In [ ]:
i = 4
x_valid = next(iter(train_dl))[1].numpy()
x_pred, z_mean, z_sd = model(x_valid[i], jr.PRNGKey(42))

df = pd.DataFrame({'prediction': x_pred[0], 'actual': x_valid[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

In [ ]:
z_mean, z_sd

In [ ]:
model.encoder(x_valid[0])

*Untrained* encoder

In [ ]:
encoder(x_valid[0])

In [ ]:
trapallada.model.image.save_vae('foo.eqx', model)